# Separable / functional factor models (track 6)

Sixth track in the factor-construction lineup, alongside vanilla /
varimax / block / sparse-warm / constrained PCA. Two ideas in one
notebook:

1. **Marginal eigenvectors of a Kronecker-shaped covariance** —
   assume ``Cov(vec(X_t)) ≈ C_expiry ⊗ C_tenor`` and extract per-axis
   eigenvector patterns whose outer product gives ready-to-use loading
   patterns on the cube. Naive moment estimator vs. flip-flop MLE.
2. **Functional PCA with a roughness penalty** — ``eigh(Σ̂ - λ·P)``
   where ``P`` is a 2D second-difference penalty, biasing the leading
   eigenvectors toward smooth loading surfaces.

Mock data only (`data/mock/atm_vol.pkl`). The numerical core lives in
`factors.separable`; this notebook walks through the API.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pca import load_long, to_wide, EXPIRY_LABELS, TENOR_LABELS
from factors import (
    marginal_kronecker_cov, kronecker_cov_mle,
    kronecker_separability_residual,
    roughness_penalty_2d, functional_pca, marginal_eigen_patterns,
    sparse_pca_warm, metrics_table,
)
from pattern_basis import (
    preset_separable_poly, patterns_to_prior_df, tensor_product_name,
)

sns.set_theme(style="whitegrid")

# Follow the sparse_pca.ipynb / factors.ipynb convention: daily diffs
# of the vol surface, computed inline from the raw long-format pkl.
vol = to_wide(load_long("../data/mock/atm_vol.pkl")).diff().dropna()
print("vol diff:", vol.shape)
n_e, n_t = len(EXPIRY_LABELS), len(TENOR_LABELS)
print(f"grid: {n_e} expiries × {n_t} tenors = {n_e * n_t} cells")

## 1. Kronecker separability: naive vs flip-flop MLE

Two estimators of ``(C_expiry, C_tenor)``. The naive one is a single
pass of normalised moments; the MLE iterates until the per-step
change in the marginals falls below `tol`. Both come out of the same
trace-normalised parameterisation so kron(C_e, C_τ) is directly
comparable to the full sample covariance.

In [ ]:
C_e_naive, C_t_naive = marginal_kronecker_cov(vol)
C_e_mle, C_t_mle, info = kronecker_cov_mle(vol, max_iter=50)
print(f"MLE: n_iter={info['n_iter']}, converged={info['converged']}")

resid_naive = kronecker_separability_residual(vol, C_e_naive, C_t_naive)
resid_mle = kronecker_separability_residual(vol, C_e_mle, C_t_mle)
print(f"separability residual  naive: {resid_naive:.4f}")
print(f"separability residual  MLE:   {resid_mle:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(C_e_mle, ax=axes[0], cmap="viridis",
            xticklabels=EXPIRY_LABELS, yticklabels=EXPIRY_LABELS)
axes[0].set_title(f"C_expiry (MLE)  trace={np.trace(C_e_mle):.2f}")
sns.heatmap(C_t_mle, ax=axes[1], cmap="viridis",
            xticklabels=TENOR_LABELS, yticklabels=TENOR_LABELS)
axes[1].set_title(f"C_tenor (MLE)  trace={np.trace(C_t_mle):.2f}")
fig.tight_layout()

**Interpretation**. A residual far from 0 means the empirical
covariance of `vec(diff_t)` cannot be matched by a single Kronecker
product — i.e. moves on this surface have cross-axis structure that
neither marginal captures alone.

On this mock the gap between the two estimators is large (`naive≈0.95
→ MLE≈0.65`): the naive one-pass estimator is heavily biased on this
surface, and even the MLE leaves enough residual to call Kronecker
separability **loose, not tight**. That puts mock vol squarely in the
"rough sketch" regime — marginal-eigvec patterns are usable as a
warm-start basis for `sparse_pca_warm` but the data-fit term needs
real weight to compensate. Watch this number on real data; if it
drops below ~0.3, the assumption is genuinely supported and you can
crank the prior `anchor` higher.

## 2. Marginal eigenvector patterns vs Legendre

Build outer-product patterns from the top eigenvectors of
``C_expiry`` and ``C_tenor`` (MLE estimate). Compare each one to the
same-named Legendre tensor-product pattern via cosine similarity —
high values mean the data-driven mode is well approximated by the
analytic level / slope / curvature shape.

Visualises the 3D surface (reusing `pattern_creator`'s ``plot_3d``
inlined here to avoid the streamlit dependency).

In [ ]:
def plot_3d_grid(arr, title, ax=None):
    """Inlined from streamlit_apps/pattern_creator.plot_3d to avoid
    pulling in streamlit."""
    show = ax is None
    if show:
        fig = plt.figure(figsize=(7, 5))
        ax = fig.add_subplot(111, projection="3d")
    X, Y = np.meshgrid(np.arange(len(TENOR_LABELS)), np.arange(len(EXPIRY_LABELS)))
    amax = max(abs(float(arr.min())), abs(float(arr.max())), 1e-9)
    surf = ax.plot_surface(X, Y, arr, cmap="RdBu_r", edgecolor="grey",
                           linewidth=0.2, vmin=-amax, vmax=amax)
    ax.set_xticks(range(len(TENOR_LABELS)))
    ax.set_xticklabels(TENOR_LABELS, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(len(EXPIRY_LABELS)))
    ax.set_yticklabels(EXPIRY_LABELS, fontsize=7)
    ax.set_xlabel("tenor"); ax.set_ylabel("expiry")
    ax.set_title(title, fontsize=10)
    return surf

N_MODES = 3  # level / slope / curvature on each axis
marginal_patterns = marginal_eigen_patterns(
    C_e_mle, C_t_mle, n_modes_e=N_MODES, n_modes_t=N_MODES,
)
legendre_patterns = preset_separable_poly(
    max_degree_expiry=N_MODES - 1, max_degree_tenor=N_MODES - 1,
)
print(f"marginal: {len(marginal_patterns)} patterns")
print(f"legendre: {len(legendre_patterns)} patterns")

grid_n = int(np.ceil(np.sqrt(len(marginal_patterns))))
fig = plt.figure(figsize=(grid_n * 4.5, grid_n * 3.5))
for idx, p in enumerate(marginal_patterns):
    ax = fig.add_subplot(grid_n, grid_n, idx + 1, projection="3d")
    plot_3d_grid(p["grid"].values, p["name"], ax=ax)
fig.suptitle("Marginal eigenvector patterns (outer products of "
             "top eigvecs of C_e_mle × top eigvecs of C_t_mle)",
             fontsize=11)
fig.tight_layout()

In [ ]:
legendre_by_name = {p["name"]: p["grid"].values.ravel()
                    for p in legendre_patterns}
marginal_by_name = {p["name"]: p["grid"].values.ravel()
                    for p in marginal_patterns}

rows = []
for name in marginal_by_name:
    if name not in legendre_by_name:
        continue
    a = marginal_by_name[name]
    b = legendre_by_name[name]
    cos = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    rows.append({"pattern": name, "abs_cosine": abs(cos), "signed": cos})
cmp = pd.DataFrame(rows).sort_values("abs_cosine", ascending=False)
print(cmp.round(3).to_string(index=False))

**Reading the table.** High `abs_cosine` says the marginal-eigvec
mode lines up with the analytic Legendre shape of the same nominal
degree (sign is incidental, eigenvectors are sign-arbitrary). On
mock vol, the top mode (`exp_level_x_ten_level`) sits at ~0.97 — a
level shape pops out of any positive covariance — while everything
else falls off fast (next highest ~0.3, most under 0.1). That means
beyond the top mode the data-driven basis isn't redundant with the
analytic Legendre prior; the two bases are largely complementary on
this surface and worth keeping side by side in the comparison
sheet.

## 3. Functional PCA λ sweep

Solves ``eigh(Σ̂ - λ·P)`` for a grid of ``λ`` values; for each, refits
on rolling windows and reports the mean ``|cosine sim|`` of the
top-3 eigenvectors against the full-sample fit. Mirrors
`factors.rolling_stability` in spirit but stays inside functional
PCA (since the existing helper always re-fits vanilla PCA). The
scale is set relative to ``||Σ̂||_F / ||P||_F`` so the same multiplier
is sensible across surfaces.

In [ ]:
def functional_rolling_stability(wide, lam, k, window=250, step=25):
    """Per-rolling-window cosine sim of functional_pca eigenvectors
    against the full-sample functional_pca eigenvectors. Same shape as
    factors.rolling_stability but specialised to functional PCA."""
    _, full_vecs = functional_pca(wide, lam=lam, k=k)
    X = wide.dropna(how="any")
    rows, dates = [], []
    for end in range(window, len(X) + 1, step):
        sub = X.iloc[end - window:end]
        try:
            _, sub_vecs = functional_pca(sub, lam=lam, k=k)
        except (np.linalg.LinAlgError, ValueError):
            continue
        num = np.abs((full_vecs * sub_vecs).sum(axis=0))
        den = (np.linalg.norm(full_vecs, axis=0)
               * np.linalg.norm(sub_vecs, axis=0))
        rows.append(num / np.where(den == 0, 1, den))
        dates.append(X.index[end - 1])
    return pd.DataFrame(
        rows, index=pd.Index(dates, name="window_end"),
        columns=[f"comp{i+1}" for i in range(k)],
    )

# Pick lam grid by tying it to ||Sigma||_F / ||P||_F so 'lam=1' means
# the penalty matches the data covariance in Frobenius scale.
flat = vol.values
Sigma = (flat.T @ flat) / flat.shape[0]
P = roughness_penalty_2d(n_e, n_t)
base = np.linalg.norm(Sigma) / np.linalg.norm(P)
lam_grid = [0.0, 0.001 * base, 0.01 * base, 0.05 * base, 0.1 * base, 0.5 * base]
print(f"base scale ||Σ||/||P|| = {base:.3e}")
print(f"lam_grid (absolute):", [f"{l:.3e}" for l in lam_grid])

stab_rows = []
for lam in lam_grid:
    stab = functional_rolling_stability(vol, lam=lam, k=3,
                                        window=250, step=25)
    stab_rows.append({"lam": lam, "mean_stability": float(stab.mean().mean()),
                      "n_windows": len(stab)})
stab_df = pd.DataFrame(stab_rows)
print(stab_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(stab_df["lam"], stab_df["mean_stability"], marker="o")
ax.set_xscale("symlog", linthresh=base * 1e-4)
ax.set_xlabel("λ (roughness penalty)")
ax.set_ylabel("mean |cosine sim|  (top-3 eigvecs)")
ax.set_title("Functional PCA stability vs λ")
fig.tight_layout()

**Reading the curve.** On this mock the curve is essentially flat
(~0.49–0.51 across the swept range) — meaning the roughness penalty
neither helps nor hurts the rolling-refit stability of the top
eigenvectors, which is also dragged down by the small number of
refit windows (`n_windows≈10` with 499 days, 250-day window, step=25).
Default to `lam=0` on mock; the right time to come back to this knob
is on real data where (i) there are more windows for a less noisy
stability estimate and (ii) a genuine smooth-vs-noisy trade-off may
appear, in which case pick the smallest λ on the high-stability
plateau.

## 4. Marginal-eigenvector patterns as a sparse-PCA prior

End-to-end demo: take the marginal-eigvec patterns from section 2,
flatten them into the prior-DataFrame shape that `sparse_pca_warm`
expects (via `pattern_basis.patterns_to_prior_df`), fit, and emit a
single-row `metrics_table` summary in the same format as every other
track. Drop that row straight into the cross-method comparison
sheet.

In [ ]:
prior = patterns_to_prior_df(marginal_patterns)
print("prior shape:", prior.shape, " (factors × cube cells)")
print("prior factors:", list(prior.index)[:5], "...")

scores, loadings, explained = sparse_pca_warm(
    vol, prior=prior, anchor=1.0, l1=0.0,
)
print("\nfit:")
print(f"  scores:    {scores.shape}")
print(f"  loadings:  {loadings.shape}")
print(f"  explained: {explained.round(4).to_dict()}")

row = metrics_table(
    name="sparse_warm_marginal_eigen",
    wide=vol, scores=scores, loadings=loadings, explained=explained,
    k_grid=(1, 3, 5),
)
print("\nmetrics_table row (drop straight into the cross-method sheet):")
print(row.round(4).to_string(index=False))

**End state.** Track 6 now has the same end-to-end story as the
other five: per-track diagnostic notebook, helper module under
`factors/`, helper tests under `tests/`, and a single-row
`metrics_table` output ready for the cross-track comparison sheet.
Next step is real data — swap `data/mock/atm_vol.pkl` for the company
pkl and re-run; the separability residual at the top will say
whether the Kronecker assumption survives the swap, and the λ sweep
in section 3 should be re-evaluated once enough rolling windows are
available.